# Reservoir time-window processing for `pandas_api`

This notebook cleans one closed UTC time window from the Aquarius reservoir exports, builds the exact `storage`/`outflow` DataFrame required by `kalmone.pandas_api.run_inflow_model`, runs the model, and visualizes the inputs and inflow estimates. Change `RESERVOIR`, `START`, and `END` in the configuration cell to process another supported window.

In [ ]:
import os
import sys
from datetime import timedelta
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots


def find_project_root() -> Path:
    explicit_root = os.environ.get("KALMONE_NOTEBOOK_ROOT")
    candidates = ([Path(explicit_root)] if explicit_root else []) + [
        Path.cwd(), *Path.cwd().parents
    ]
    for candidate in candidates:
        has_data = (candidate / "Reservoirs").is_dir()
        has_package = (candidate / "src" / "kalmone").is_dir()
        if has_data and has_package:
            return candidate
    raise FileNotFoundError(
        "Could not find the project; launch inside it or set KALMONE_NOTEBOOK_ROOT."
    )


ROOT = find_project_root()
for import_root in [
    ROOT / "src", ROOT / "Notebooks" / "reservoir_pandas_api"
]:
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from prepare_reservoir_data import (  # noqa: E402
    prepare_reservoir_data,
    sources_for_reservoir,
)

from kalmone import CFS_TO_ACRE_FEET_PER_SECOND  # noqa: E402
from validation import (  # noqa: E402
    ValidationSettings,
    generate_validation_outputs,
)
from kalmone.pandas_api import run_inflow_model  # noqa: E402

## 1. Select a reservoir and UTC time window

Both endpoints are included. The timestamps must include a timezone; they are converted to UTC by the processing module. The noise values below match the illustrative settings in the streaming benchmark and should be reviewed before operational use.

In [ ]:
RESERVOIR = "Chesbro"  # Also supports "Chesbro".
START = pd.Timestamp("1982-11-01 00:00:00", tz="UTC")
END = pd.Timestamp("1983-04-01 00:00:00", tz="UTC")
RESERVOIR = "Lexington"  # Also supports "Chesbro".
START = pd.Timestamp("2023-11-01 00:00:00", tz="UTC")
END = pd.Timestamp("2025-04-01 00:00:00", tz="UTC")
ASOF_TOLERANCE = pd.Timedelta("20min")
ROLLING_WINDOW = "6h"
SMOOTHING_LAG = timedelta(hours=4)
VALIDATION_FREQUENCY = "1h"
CROSS_CORRELATION_RANGE = 48
MINIMUM_PAIRED_OBSERVATIONS = 24
TRAINING_START = None
TRAINING_END = None
EVALUATION_START = None
EVALUATION_END = None

Q_STORAGE = 0.002
Q_INFLOW = 3
Q_OUTFLOW = 0.05
R_STORAGE = 10.0**2
R_OUTFLOW = 20.0**2

Q_STORAGE = 2.4516645375091118e-12
Q_INFLOW = 34.562596569192635
Q_OUTFLOW = 0.0002268274627381655
R_STORAGE = 2.6371394307731743e-13
R_OUTFLOW = 0.00018248750752536665

print(f"Reservoir: {RESERVOIR}")
print(f"Window: {START} through {END}")

## 2. Clean, align, and audit the source data

Storage timestamps define the model clock. Outlet, spillway, and upstream observations are aligned causally with a backward as-of match, so no future measurement is used. `prepared.observations` contains only the two columns required by `pandas_api`; `prepared.diagnostics` retains the component series for plotting and review.

In [ ]:
sources = sources_for_reservoir(ROOT, RESERVOIR)
prepared = prepare_reservoir_data(
    sources, start=START, end=END, asof_tolerance=ASOF_TOLERANCE
)
observations = prepared.observations

display(prepared.source_audit[[
    "path", "source_rows", "clean_rows", "duplicates_removed",
    "finite_value_fraction", "first_utc", "last_utc",
]])
display(prepared.window_audit)
display(observations.head())
display(observations.isna().mean().rename("missing_fraction").to_frame())

## 3. Run the pandas API and build comparison series

The raw water-balance calculation uses the actual elapsed time between observations. Its centered rolling mean is a simple acausal baseline because it uses values on both sides of each timestamp. `estimated_inflow` is the immediate filtered estimate; `revised_inflow` is the delayed fixed-lag replacement.

In [ ]:
model_output = run_inflow_model(
    observations,
    q_storage=Q_STORAGE,
    q_inflow=Q_INFLOW,
    q_outflow=Q_OUTFLOW,
    r_storage=R_STORAGE,
    r_outflow=R_OUTFLOW,
    smoothing_lag=SMOOTHING_LAG,
)

dt_seconds = observations.index.to_series().diff().dt.total_seconds()
raw_inflow = (
    observations["storage"].diff()
    / (CFS_TO_ACRE_FEET_PER_SECOND * dt_seconds)
    + observations["outflow"]
).rename("raw_inflow")
centered_rolling_inflow = raw_inflow.rolling(
    ROLLING_WINDOW, min_periods=2, center=True
).mean().rename(
    "centered_rolling_inflow"
)
comparison = prepared.diagnostics.join(model_output, how="left")
comparison = comparison.join(raw_inflow).join(centered_rolling_inflow)

display(model_output.head())
display(comparison[[
    "raw_inflow", "centered_rolling_inflow", "estimated_inflow", "revised_inflow"
]].describe().T)

## 4. Visualize model inputs

Storage and flow use separate panels because their units differ. The rangeslider supports close inspection of gaps and rapid changes within the selected window.

In [ ]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
    subplot_titles=("Storage input", "Aligned outflow components"),
)
fig.add_trace(go.Scatter(
    x=comparison.index, y=comparison["storage"], mode="lines", name="Storage",
    line={"color": "#7f7f7f", "width": 1.8}, connectgaps=False,
), row=1, col=1)
for column, label, color in [
    ("outlet_discharge", "Outlet discharge", "#ff7f0e"),
    ("spillway_flow", "Spillway flow", "#2ca02c"),
    ("outflow", "Model outflow", "#1f77b4"),
]:
    fig.add_trace(go.Scatter(
        x=comparison.index, y=comparison[column], mode="lines", name=label,
        line={"color": color, "width": 1.5}, connectgaps=False,
    ), row=2, col=1)
fig.update_yaxes(title_text="Storage (acre-ft)", row=1, col=1)
fig.update_yaxes(title_text="Flow (cfs)", row=2, col=1)
fig.update_xaxes(
    title_text="UTC timestamp", rangeslider={"visible": True}, row=2, col=1
)
fig.update_layout(
    title=f"{RESERVOIR}: cleaned pandas_api inputs", template="plotly_white",
    hovermode="x unified", height=700,
)
fig.show()

## 5. Compare inflow estimates

The upstream series is a partial-catchment proxy, not total-inflow truth. The raw calculation exposes storage-differencing noise; the rolling mean and Kalmone outputs show two different smoothing approaches.

In [ ]:
fig = go.Figure()
for column, label, color, width, opacity, dash in [
    ("raw_inflow", "Raw water balance", "#d62728", 0.8, 0.35, "solid"),
    (
        "centered_rolling_inflow", f"{ROLLING_WINDOW} centered mean",
        "#ff7f0e", 1.4, 0.9, "solid",
    ),
    ("estimated_inflow", "Kalmone causal inflow", "#1f77b4", 2.0, 1.0, "solid"),
    (
        "revised_inflow", f"Kalmone revised ({SMOOTHING_LAG})",
        "#2ca02c", 1.8, 1.0, "dash",
    ),
    ("upstream_flow", "Upstream proxy", "#9467bd", 1.2, 0.8, "solid"),
]:
    fig.add_trace(go.Scattergl(
        x=comparison.index, y=comparison[column], mode="lines", name=label,
        connectgaps=False, opacity=opacity,
        line={"color": color, "width": width, "dash": dash},
        hovertemplate="%{y:,.2f} cfs<extra>" + label + "</extra>",
    ))
fig.add_hline(y=0, line_width=1, line_color="gray")
fig.update_layout(
    title=f"{RESERVOIR}: inflow comparison", template="plotly_white",
    xaxis={"title": "UTC timestamp", "rangeslider": {"visible": True}},
    yaxis_title="Flow (cfs)", hovermode="x unified", height=580,
    margin={"l": 75, "r": 230, "t": 80, "b": 55},
    legend={"orientation": "v", "x": 1.02, "y": 1.0},
)
fig.show()

## 6. Validate against the upstream proxy and storage closure

All comparisons are resampled to hourly means without interpolation. The upstream gauge is a partial-catchment proxy, so agreement metrics describe proxy agreement and timing rather than total-inflow accuracy. Storage closure remains the stronger internal-consistency check. The revised series is retained only as a delayed diagnostic.

In [ ]:
validation_settings = ValidationSettings(
    evaluation_frequency=VALIDATION_FREQUENCY,
    cross_correlation_range=CROSS_CORRELATION_RANGE,
    minimum_paired_observations=MINIMUM_PAIRED_OBSERVATIONS,
    centered_rolling_window=ROLLING_WINDOW,
    training_start=TRAINING_START,
    training_end=TRAINING_END,
    evaluation_start=EVALUATION_START,
    evaluation_end=EVALUATION_END,
)
validation_outputs = generate_validation_outputs(
    comparison, settings=validation_settings, make_plots=True
)

display(validation_outputs.upstream_proxy_agreement)
display(validation_outputs.best_lag_summary)
display(validation_outputs.storage_closure)

In [ ]:
validation_outputs.cross_correlation_plot.show()
validation_outputs.estimate_upstream_scatter_plot.show()